# Figures de l'article — disconnectome, graphe, cerveau

Ce notebook génère **6 figures** (PDF vectoriel + PNG 300 dpi) dans `nemo_results/figures_article/` :

| Fichier | Contenu | Place dans l'article |
|---|---|---|
| `fig_atlas` | Atlas CocoYeo143 sur le cerveau + répartition des régions par réseau | Méthodes 3.2 |
| `fig_lesion_frequency` | Carte de fréquence des 127 lésions (MNI) | Résultats / cohorte |
| `fig_patient_pipeline` | Un patient : lésion → matrice ChaCo → graphe | Méthodes 3.4 |
| `fig_chaco_groups` | ChaCo moyen favorable / défavorable / différence | Résultats |
| `fig_shap_regions` | Importance SHAP des 143 régions sur le cerveau | Résultats 4.5 |
| `fig_degree_distribution` | Distribution du nombre d'arêtes par graphe | Méthodes (option) |

**Rien à modifier** : exécute *Exécution → Tout exécuter*. Seule option : `EXAMPLE_PID` (patient de la figure 3).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q nilearn torch_geometric

In [ ]:
# ============================ PARAMÈTRES ======================================
import os
BASE     = "/content/drive/MyDrive/isles_prp/MNI"
NEMO     = os.path.join(BASE, "nemo_results")
PT_PATH  = os.path.join(NEMO, "pyg_dataset.pt")
NAMES    = os.path.join(NEMO, "region_names.txt")
SHAP_CSV = os.path.join(NEMO, "shap_regions_v2.csv")
FIG_DIR  = os.path.join(NEMO, "figures_article")
EXAMPLE_PID = None      # ex. "sub-stroke0189" ; None = choix automatique
TOP_PERCENT = 0.15      # même seuil que le dataset
# ==============================================================================
os.makedirs(FIG_DIR, exist_ok=True)
for p in [PT_PATH, NAMES, SHAP_CSV]:
    print(("OK      " if os.path.exists(p) else "MANQUANT"), p)

In [ ]:
import torch_geometric.typing  # évite l'import circulaire connu
import re, io, glob, zipfile, pickle, warnings
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
import matplotlib as mpl
from nilearn import datasets, plotting, image
warnings.filterwarnings("ignore")
plt.rcParams.update({"font.size": 9, "pdf.fonttype": 42, "font.family": "DejaVu Sans"})

def save(fig, name):
    for ext in ("pdf", "png"):
        fig.savefig(os.path.join(FIG_DIR, f"{name}.{ext}"), dpi=300, bbox_inches="tight")
    print("Figure enregistrée :", name)

# ---------- données ----------
dataset = torch.load(PT_PATH, weights_only=False)
names = [l.strip() for l in open(NAMES, encoding="utf-8") if l.strip()]
assert len(names) == 143, len(names)
pids = [d.pid for d in dataset]
y    = np.array([int(d.y) for d in dataset])
CHACO = np.vstack([d.x[:, 0].numpy() for d in dataset])          # (127,143) ChaCo régional brut
print(f"{len(dataset)} patients | favorables={int((y==0).sum())} défavorables={int((y==1).sum())}")
print("ChaCo min/max :", CHACO.min().round(3), CHACO.max().round(3))
print("Exemples de noms :", names[:2], names[16:18], names[60:62])

## 1. Réseau de chaque région + coordonnées (centroïdes MNI)

In [ ]:
def network_of(n):
    if n.startswith("Cbm_"): return "Cerebellum"
    if any(s in n for s in ("Thalamus","Caudate","Putamen","Pallidum","Hippocampus",
                            "Amygdala","Accumbens","VentralDC")): return "Subcortical"
    m = re.search(r"7Networks_(?:LH_|RH_)?([A-Za-z]+)", n)   # "lh.7Networks_Vis_1" ou "7Networks_LH_Vis_1"
    return {"Vis":"Visual","SomMot":"Somatomotor","DorsAttn":"Dorsal attention",
            "SalVentAttn":"Salience/Ventral attn.","Limbic":"Limbic",
            "Cont":"Frontoparietal control","Default":"Default mode"}.get(m.group(1),"Other") if m else "Other"

NET = np.array([network_of(n) for n in names])
NET_ORDER = ["Visual","Somatomotor","Dorsal attention","Salience/Ventral attn.","Limbic",
             "Frontoparietal control","Default mode","Subcortical","Cerebellum"]
NET_COLOR = {  # couleurs standard de Yeo 2011 + gris pour sous-cortical / cervelet
 "Visual":"#781286","Somatomotor":"#4682B4","Dorsal attention":"#00760E",
 "Salience/Ventral attn.":"#C43AFA","Limbic":"#B5C97A","Frontoparietal control":"#E69422",
 "Default mode":"#CD3E4E","Subcortical":"#555555","Cerebellum":"#A0785A"}
print(pd.Series(NET).value_counts())
assert (NET == "Other").sum() == 0, [n for n, k in zip(names, NET) if k == "Other"]

In [ ]:
def side_of(n):
    s = n.lower()
    if re.search(r"(^|[^a-z])(left|lh|l)([^a-z]|$)", s): return "L"
    if re.search(r"(^|[^a-z])(right|rh|r)([^a-z]|$)", s): return "R"
    return "M"
norm = lambda s: re.sub(r"[^a-z0-9]", "", s.lower())
COORDS = np.full((143, 3), np.nan)

# --- (a) Schaefer 100 x 7 : correspondance exacte par nom ---
sch = datasets.fetch_atlas_schaefer_2018(n_rois=100, yeo_networks=7, resolution_mm=1)
sch_labels = [l.decode() if isinstance(l, bytes) else l for l in sch.labels]
if sch_labels[0].lower() == "background": sch_labels = sch_labels[1:]
c_sch, val_sch = plotting.find_parcellation_cut_coords(sch.maps, return_label_names=True)
sch_xyz = {sch_labels[int(v) - 1]: c for c, v in zip(c_sch, val_sch)}
def to_schaefer(n):
    """'lh.7Networks_Vis_1' -> '7Networks_LH_Vis_1' (format nilearn) ; inchangé sinon."""
    m = re.match(r"^(lh|rh)\.7Networks_(.+)$", n, re.I)
    return f"7Networks_{m.group(1).upper()}_{m.group(2)}" if m else n
for i, n in enumerate(names):
    key = to_schaefer(n)
    if key in sch_xyz: COORDS[i] = sch_xyz[key]
    else:
        for lab, c in sch_xyz.items():
            if key.endswith(lab): COORDS[i] = c; break

# --- (b) Sous-cortical : Harvard-Oxford (repli : centroïdes MNI usuels) ---
SUB_FALLBACK = {"Thalamus":(11,-18,7),"Caudate":(13,12,9),"Putamen":(25,2,0),"Pallidum":(19,-3,-1),
                "Hippocampus":(26,-21,-13),"Amygdala":(23,-4,-19),"Accumbens":(9,11,-7),"VentralDC":(12,-17,-9)}
ho_xyz = {}
try:
    ho = datasets.fetch_atlas_harvard_oxford("sub-maxprob-thr25-1mm")
    c_ho, v_ho = plotting.find_parcellation_cut_coords(ho.maps, return_label_names=True)
    for c, v in zip(c_ho, v_ho):
        ho_xyz[ho.labels[int(v)]] = c
except Exception as e:
    print("Harvard-Oxford indisponible, repli sur centroïdes usuels :", e)
for i, n in enumerate(names):
    if NET[i] != "Subcortical": continue
    struct = next(s for s in SUB_FALLBACK if s in n)
    sd = side_of(n); key = f"{'Left' if sd=='L' else 'Right'} {struct}"
    if key in ho_xyz: COORDS[i] = ho_xyz[key]
    else:
        x, yy, z = SUB_FALLBACK[struct]; COORDS[i] = (-x if sd == "L" else x, yy, z)

# --- (c) Cervelet : atlas SUIT (Diedrichsen 2009) ---
URL = "https://raw.githubusercontent.com/DiedrichsenLab/cerebellar_atlases/master/Diedrichsen_2009/"
!wget -q -nc {URL}atl-Anatom_space-MNI_dseg.nii {URL}atl-Anatom.tsv
suit_tsv = pd.read_csv("atl-Anatom.tsv", sep="\t")
c_suit, v_suit = plotting.find_parcellation_cut_coords("atl-Anatom_space-MNI_dseg.nii", return_label_names=True)
suit_xyz = {norm(suit_tsv.set_index("index").loc[int(v), "name"]): c for c, v in zip(c_suit, v_suit)}
for i, n in enumerate(names):
    if NET[i] != "Cerebellum": continue
    k = norm(n.replace("Cbm_", ""))
    if k in suit_xyz: COORDS[i] = suit_xyz[k]

miss = [names[i] for i in range(143) if np.isnan(COORDS[i]).any()]
print(f"Coordonnées trouvées : {143 - len(miss)}/143")
if miss: print("Sans coordonnées (exclues des cartes cérébrales) :", miss)
OK = ~np.isnan(COORDS).any(1)
pd.DataFrame({"region": names, "network": NET, "x": COORDS[:,0], "y": COORDS[:,1], "z": COORDS[:,2]}
            ).to_csv(os.path.join(FIG_DIR, "region_centroids.csv"), index=False)

## Figure A — Atlas CocoYeo143 et répartition des régions

In [ ]:
fig = plt.figure(figsize=(7.2, 4.6))
ax1 = fig.add_axes([0.0, 0.40, 1.0, 0.60])
ax2 = fig.add_axes([0.10, 0.04, 0.85, 0.30])
cols = [NET_COLOR[k] for k in NET[OK]]
plotting.plot_connectome(np.zeros((OK.sum(), OK.sum())), COORDS[OK], node_color=cols,
                         node_size=22, display_mode="lzry", axes=ax1, figure=fig,
                         annotate=False, colorbar=False, node_kwargs={"edgecolor": "k", "linewidths": 0.3})
ax1.set_title("a   CocoYeo143 atlas (143 regions)", loc="left", fontsize=9, fontweight="bold")
counts = pd.Series(NET).value_counts().reindex(NET_ORDER)
ax2.bar(range(len(counts)), counts.values, color=[NET_COLOR[k] for k in NET_ORDER], edgecolor="k", lw=0.4)
for k, v in enumerate(counts.values): ax2.text(k, v + 0.5, str(v), ha="center", fontsize=7)
ax2.set_xticks(range(len(counts))); ax2.set_xticklabels(NET_ORDER, rotation=30, ha="right", fontsize=7)
ax2.set_ylabel("Number of regions"); ax2.spines[["top","right"]].set_visible(False)
ax2.set_title("b   Regions per network", loc="left", fontsize=9, fontweight="bold")
save(fig, "fig_atlas"); plt.show()

## Figure B — Carte de fréquence des lésions (n = 127)

In [ ]:
tpl = datasets.load_mni152_template(resolution=1)
acc, n_ok = None, 0
for pid in pids:
    f = os.path.join(BASE, pid, "registered", f"{pid}_lesion_MNI.nii.gz")
    if not os.path.exists(f):
        cand = glob.glob(os.path.join(BASE, pid, "**", "*lesion*MNI*.nii*"), recursive=True)
        if not cand: print("masque introuvable :", pid); continue
        f = cand[0]
    m = image.resample_to_img(f, tpl, interpolation="nearest")
    a = (m.get_fdata() > 0).astype(np.float32)
    acc = a if acc is None else acc + a; n_ok += 1
freq = image.new_img_like(tpl, acc)
print(f"{n_ok} masques cumulés | recouvrement max = {int(acc.max())} patients")

fig = plt.figure(figsize=(7.2, 1.9))
plotting.plot_stat_map(freq, bg_img=tpl, display_mode="z", cut_coords=[-30,-15,-5,5,15,30,45],
                       cmap="hot", threshold=0.5, vmax=acc.max(), colorbar=True, figure=fig,
                       annotate=True, black_bg=False, draw_cross=False)
fig.text(0.01, 1.0, f"Lesion overlap (number of patients, n = {n_ok})", fontsize=9, fontweight="bold")
save(fig, "fig_lesion_frequency"); plt.show()

## Figure C — Du masque lésionnel au graphe (un patient)

In [ ]:
def load_conn(pid):
    """Matrice chacoconn 143x143 du patient (lue dans les zips NeMo)."""
    for zf in sorted(glob.glob(os.path.join(NEMO, "*.zip"))):
        with zipfile.ZipFile(zf) as z:
            for mbr in z.namelist():
                b = os.path.basename(mbr).lower()
                if pid in b and "chacoconn" in b and b.endswith("mean.pkl"):
                    obj = pickle.load(io.BytesIO(z.read(mbr)))
                    A = obj.toarray() if hasattr(obj, "toarray") else np.asarray(obj)
                    A = np.maximum(A, A.T); np.fill_diagonal(A, 0); return A
    raise FileNotFoundError(pid)

if EXAMPLE_PID is None:   # patient défavorable à charge de déconnexion médiane
    idx1 = np.where(y == 1)[0]; tot = CHACO[idx1].sum(1)
    EXAMPLE_PID = pids[idx1[np.argsort(tot)[len(tot) // 2]]]
k = pids.index(EXAMPLE_PID); d = dataset[k]
A = load_conn(EXAMPLE_PID)
print("Patient exemple :", EXAMPLE_PID, "| mRS>2 :", bool(y[k]), "| arêtes :", d.edge_index.shape[1] // 2)

fig = plt.figure(figsize=(7.4, 6.4))
# (a) lésion
axa = fig.add_axes([0.00, 0.60, 0.50, 0.38])
les = glob.glob(os.path.join(BASE, EXAMPLE_PID, "**", "*lesion*MNI*.nii*"), recursive=True)[0]
les = image.resample_to_img(les, tpl, interpolation="nearest")
plotting.plot_roi(les, bg_img=tpl, display_mode="ortho", cut_coords=plotting.find_xyz_cut_coords(les),
                  cmap="autumn", axes=axa, figure=fig, draw_cross=False, annotate=False, black_bg=False)
axa.set_title("a   Lesion mask (MNI)", loc="left", fontsize=9, fontweight="bold")
# (b) matrice ChaCo ordonnée par réseau
axb = fig.add_axes([0.56, 0.58, 0.40, 0.40])
order = np.concatenate([np.where(NET == n)[0] for n in NET_ORDER])
im = axb.imshow(A[np.ix_(order, order)], cmap="magma", vmin=0, vmax=max(np.percentile(A[A > 0], 99), 1e-3))
bounds = np.cumsum([np.sum(NET == n) for n in NET_ORDER])[:-1] - 0.5
for b_ in bounds: axb.axhline(b_, c="w", lw=0.4); axb.axvline(b_, c="w", lw=0.4)
axb.set_xticks([]); axb.set_yticks([]); axb.set_xlabel("Regions (ordered by network)", fontsize=7)
plt.colorbar(im, ax=axb, fraction=0.046, pad=0.03).set_label("Pairwise ChaCo", fontsize=7)
axb.set_title("b   Disconnection matrix (143 × 143)", loc="left", fontsize=9, fontweight="bold")
# (c) graphe seuillé
axc = fig.add_axes([0.00, 0.02, 1.0, 0.50])
ei, ew = d.edge_index.numpy(), d.edge_attr.squeeze(-1).numpy()
G = np.zeros((143, 143)); G[ei[0], ei[1]] = ew
cmap = mpl.cm.YlOrRd; vmax = max(CHACO[k].max(), 1e-3)
ncol = [cmap(v / vmax) for v in CHACO[k][OK]]
nsize = 8 + 60 * CHACO[k][OK] / vmax
plotting.plot_connectome(G[np.ix_(OK, OK)], COORDS[OK], node_color=ncol, node_size=nsize,
                         edge_cmap="Greys", edge_kwargs={"linewidth": 0.5, "alpha": 0.5},
                         display_mode="lzry", axes=axc, figure=fig, annotate=False, colorbar=False,
                         node_kwargs={"edgecolor": "k", "linewidths": 0.2})
axc.set_title(f"c   Patient graph: nodes = regional ChaCo (colour, size), edges = top {int(TOP_PERCENT*100)}% pairwise ChaCo",
              loc="left", fontsize=9, fontweight="bold")
save(fig, "fig_patient_pipeline"); plt.show()

## Figure D — ChaCo régional moyen : favorable vs défavorable

In [ ]:
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
m0, m1 = CHACO[y == 0].mean(0), CHACO[y == 1].mean(0)
pv = np.array([mannwhitneyu(CHACO[y == 1, j], CHACO[y == 0, j]).pvalue if CHACO[:, j].any() else 1.0
               for j in range(143)])
q = multipletests(pv, method="fdr_bh")[1]
pd.DataFrame({"region": names, "network": NET, "mean_fav": m0, "mean_unfav": m1, "diff": m1 - m0,
              "p": pv, "q_fdr": q}).sort_values("p").to_csv(os.path.join(FIG_DIR, "chaco_group_stats.csv"), index=False)
print(f"Régions significatives (FDR q<0.05) : {(q < 0.05).sum()} / 143")

vmax = max(m0.max(), m1.max()); dmax = np.abs(m1 - m0).max()
fig = plt.figure(figsize=(7.2, 6.0))
panels = [(m0, "a   Favorable outcome (mRS ≤ 2)", "YlOrRd", 0, vmax),
          (m1, "b   Unfavorable outcome (mRS > 2)", "YlOrRd", 0, vmax),
          (m1 - m0, "c   Difference (unfavorable − favorable)", "RdBu_r", -dmax, dmax)]
for r, (vals, title, cm, lo, hi) in enumerate(panels):
    ax = fig.add_axes([0.0, 0.67 - r * 0.33, 1.0, 0.31])
    plotting.plot_markers(vals[OK], COORDS[OK], node_size=10 + 70 * np.abs(vals[OK]) / max(np.abs(vals).max(), 1e-6),
                          node_cmap=cm, node_vmin=lo, node_vmax=hi, display_mode="lzry",
                          axes=ax, figure=fig, annotate=False, colorbar=True)
    ax.set_title(title, loc="left", fontsize=9, fontweight="bold")
save(fig, "fig_chaco_groups"); plt.show()

## Figure E — Importance SHAP des 143 régions

In [ ]:
sh = pd.read_csv(SHAP_CSV)
imp = np.zeros(143); sgn = np.zeros(143)
pos = {n: i for i, n in enumerate(names)}
for _, r in sh.iterrows():
    i = pos[r["region"]]; imp[i] = r["shap_abs"]; sgn[i] = np.sign(r["spearman_sens"])
signed = imp * sgn
m = np.abs(signed).max()
fig = plt.figure(figsize=(7.2, 2.6))
plotting.plot_markers(signed[OK], COORDS[OK], node_size=4 + 90 * imp[OK] / imp.max(),
                      node_cmap="RdBu_r", node_vmin=-m, node_vmax=m, display_mode="lzry",
                      figure=fig, annotate=False, colorbar=True)
fig.text(0.01, 1.0, "Regional SHAP importance (size = mean |SHAP|; red: disconnection → unfavorable, blue: → favorable)",
         fontsize=9, fontweight="bold")
save(fig, "fig_shap_regions"); plt.show()
print(sh.head(10)[["rang", "region", "reseau", "shap_abs", "shap_std", "spearman_sens"]])

## Figure F (option) — Taille des graphes et charge de déconnexion

In [ ]:
nedges = np.array([d.edge_index.shape[1] // 2 for d in dataset])
tot = CHACO.sum(1)
fig, ax = plt.subplots(1, 2, figsize=(7.2, 2.5))
ax[0].hist(nedges, bins=25, color="#4682B4", edgecolor="k", lw=0.3)
ax[0].set_xlabel("Edges per graph (after top-15% threshold)"); ax[0].set_ylabel("Patients")
ax[0].set_title("a", loc="left", fontweight="bold")
bp = ax[1].boxplot([tot[y == 0], tot[y == 1]], patch_artist=True, widths=0.5)
ax[1].set_xticks([1, 2]); ax[1].set_xticklabels(["mRS ≤ 2", "mRS > 2"])
for p_, c in zip(bp["boxes"], ["#9ecae1", "#fc9272"]): p_.set_facecolor(c)
ax[1].set_ylabel("Total disconnection (Σ ChaCo)"); ax[1].set_title("b", loc="left", fontweight="bold")
for a in ax: a.spines[["top","right"]].set_visible(False)
print("Mann-Whitney Σ ChaCo : p =", round(mannwhitneyu(tot[y == 1], tot[y == 0]).pvalue, 4))
save(fig, "fig_degree_distribution"); plt.show()

In [ ]:
!cd "{FIG_DIR}" && zip -q -r ../figures_article.zip . && ls -la
print("Archive :", os.path.join(NEMO, "figures_article.zip"))